# Phase 1 — Baseline evaluation ("the before photo")

**What this notebook does:** measures how many of the 164 standard buggy programs (HumanEvalFix) the *untouched* model can already fix. Without this number, we can never prove our teaching helped.

**GPU required: L4** (Runtime → Change runtime type → L4). The T4 cannot do bf16 math and would break the frozen protocol.

**Cost estimate:** Stage A ~10 min. Stage B ~2–4 hours (~10–20 compute units). Keep the tab open.

**Two stages:**
- **Stage A (cheap rehearsal):** 20 problems, 1 attempt each — just to check the model's raw outputs look sane before paying for the full run. *You read the outputs and decide.*
- **Stage B (the real measurement):** all 164 problems × 20 attempts each, graded by running the tests.

In [ ]:
# 1) Check we have the right GPU
!nvidia-smi
import torch
print(torch.cuda.get_device_name(0))
assert torch.cuda.is_bf16_supported(), "This GPU cannot do bf16 — switch runtime to L4 (Runtime > Change runtime type)."

In [ ]:
# 2) Connect Google Drive (results survive a disconnect)
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/rl-post-training/phase1'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Results will sync to:', RESULTS_DIR)

In [ ]:
# 3) Install the paper's own grading harness and RECORD ITS VERSION
#    (the commit hash goes into EVAL_PROTOCOL.md — it is part of the frozen ruler)
%cd /content
!git clone -q https://github.com/bigcode-project/bigcode-evaluation-harness.git
%cd /content/bigcode-evaluation-harness
HARNESS_COMMIT = !git rev-parse HEAD
HARNESS_COMMIT = HARNESS_COMMIT[0]
print('HARNESS COMMIT (copy into EVAL_PROTOCOL.md):', HARNESS_COMMIT)
with open(os.path.join(RESULTS_DIR, 'harness_commit.txt'), 'w') as f:
    f.write(HARNESS_COMMIT)
!pip install -q -e .
!pip install -q -U transformers accelerate

## Configuration

First run: leave as is (Qwen). Second run (later): change `MODEL` to `meta-llama/Llama-3.2-3B-Instruct`, set `BATCH = 8`, change `RUN_NAME` to `llama_base`, and run the login cell at the bottom first (Llama is gated).

In [ ]:
MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
TASK = 'humanevalfixtests-python'
PROMPT = 'instruct'   # the prompt format under test in Stage A; freeze the winner
BATCH = 16
RUN_NAME = 'qwen_base'
print(MODEL, '|', TASK, '| prompt =', PROMPT)

## Stage A — cheap rehearsal (20 problems, 1 attempt, ~10 min)

We check the model's raw outputs *before* paying for the full run. A formatting problem here would silently ruin the whole measurement (the grader can't extract code that isn't where it expects).

In [ ]:
%cd /content/bigcode-evaluation-harness
!accelerate launch main.py \
  --model {MODEL} --tasks {TASK} --prompt {PROMPT} \
  --do_sample True --temperature 0.2 --top_p 0.95 --n_samples 1 \
  --batch_size {BATCH} --precision bf16 --limit 20 \
  --max_length_generation 2048 --allow_code_execution \
  --save_generations --save_generations_path /content/smoke_gens.json \
  --metric_output_path /content/smoke_metrics.json

In [ ]:
# READ THE RAW OUTPUTS — this is the decision point.
# Healthy = a complete rewritten function that fixes the bug.
# Sick    = empty output, chat rambling, repeating the question, cut-off code.
import glob, json
path = sorted(glob.glob('/content/smoke_gens*.json'))[0]
gens = json.load(open(path))
for i in range(3):
    print('=' * 70)
    print(f'PROBLEM {i} — model output:')
    print(gens[i][0][:2500])
print('=' * 70)
print(open('/content/smoke_metrics.json').read())

**If the outputs look sick:** change `PROMPT` in the config cell (the harness supports several formats for this task — see its `docs/` folder), rerun Stage A, and compare. Freeze whichever format produces clean outputs — that choice gets written into `EVAL_PROTOCOL.md` and never changes again.

**If the outputs look healthy:** proceed to Stage B.

## Stage B — the real measurement (164 problems × 20 attempts, ~2–4 h)

Everything here is the frozen protocol: temperature 0.2, top_p 0.95, n=20, grading by running the tests. Results save to Drive as they finish.

In [ ]:
GEN_PATH = os.path.join(RESULTS_DIR, f'gens_{RUN_NAME}.json')
MET_PATH = os.path.join(RESULTS_DIR, f'metrics_{RUN_NAME}.json')
%cd /content/bigcode-evaluation-harness
!accelerate launch main.py \
  --model {MODEL} --tasks {TASK} --prompt {PROMPT} \
  --do_sample True --temperature 0.2 --top_p 0.95 --n_samples 20 \
  --batch_size {BATCH} --precision bf16 \
  --max_length_generation 2048 --allow_code_execution \
  --save_generations --save_generations_path {GEN_PATH} \
  --metric_output_path {MET_PATH}

In [ ]:
# The before-photo number
print(open(MET_PATH).read())
print('\nSaved to Drive:', GEN_PATH)

## Bring back to the session
1. The **pass@1 number** from the metrics above
2. The **harness commit hash** (printed in step 3)
3. **2–3 raw outputs from Stage A** (paste them) — healthy or sick, we read them together

Then we lock the target and freeze `EVAL_PROTOCOL.md`.

---
### Only for the Llama run (second pass, later)
Llama-3.2 is gated — log in with your Hugging Face token first, then rerun from the config cell with the Llama settings.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()